# Day 9 · Exercise 3: Few-Shot Address Extractor

**What you'll build:** `extract_address_few_shot(text: str, model: str) -> dict` — a function that injects two hand-written (input prose, output JSON) example pairs into the messages list before asking the model to extract address fields from real text.

**Why it matters:** Adding even one worked example to the conversation is the single highest-leverage accuracy lever in structured extraction — it resolves format ambiguity, anchors null-versus-omit behaviour, and consistently improves field coverage over schema descriptions alone.

## Your Implementation

In [ ]:
import json
import ollama
from pydantic import BaseModel, Field


class AddressInfo(BaseModel):
    street: str | None = Field(
        default=None,
        description="Street address including number and street name, or null if not present",
    )
    city: str | None = Field(
        default=None,
        description="City or town name, or null if not present",
    )
    state: str | None = Field(
        default=None,
        description="State, province, or region name, or null if not present",
    )
    country: str = Field(
        description="Country name — always present; infer from context if needed",
    )
    postcode: str | None = Field(
        default=None,
        description="Postal or ZIP code as a string, or null if not present",
    )


def extract_address_few_shot(text: str, model: str) -> dict:
    """Extract address fields from prose text using two few-shot example pairs.

    Builds a messages list containing:
      - A system message with the AddressInfo JSON Schema and extraction rules.
      - Two user/assistant demonstration pairs (fabricated, not real data).
      - A final user message containing the real text to extract from.

    Args:
        text:  The prose input to extract an address from.
        model: Ollama model name (e.g. "llama3.2").

    Returns:
        A plain dict with keys: street, city, state, country, postcode.
        Optional fields are None when absent in the text.

    Example:
        result = extract_address_few_shot(
            "Send the parcel to 42 Maple Drive, Austin, Texas 78701, USA.",
            model="llama3.2",
        )
        # result["city"] == "Austin"
        # result["postcode"] == "78701"
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'


def _run_checks():
    score, total = 0, 4

    # Check 1: function exists and is callable
    try:
        assert callable(extract_address_few_shot), 'extract_address_few_shot is not defined'
        print(f'{_PASS} Check 1/{total}: function exists and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: returns a dict for a fully described address
    try:
        result = extract_address_few_shot(
            "Please deliver to 7 Harbour View Road, Cape Town, Western Cape, 8001, South Africa.",
            model="llama3.2",
        )
        assert isinstance(result, dict), f'expected dict, got {type(result).__name__}'
        print(f'{_PASS} Check 2/{total}: returns a dict')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')
        return

    # Check 3: required field country is present and non-empty
    try:
        assert 'country' in result, "key 'country' missing from result"
        assert result['country'], "country is empty or None"
        print(f'{_PASS} Check 3/{total}: required field "country" is present and non-empty')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: optional fields default to None (not hallucinated) for sparse input
    try:
        sparse = extract_address_few_shot(
            "I live in Tokyo, Japan.",
            model="llama3.2",
        )
        assert isinstance(sparse, dict), f'expected dict for sparse input, got {type(sparse).__name__}'
        assert 'country' in sparse, "key 'country' missing from sparse result"
        # street should be None because it was not mentioned
        street_val = sparse.get('street')
        assert street_val is None, (
            f'expected street=None for sparse input, got {street_val!r} — '
            'check that your example pair teaches the model to return null for absent fields'
        )
        print(f'{_PASS} Check 4/{total}: optional fields are None when absent (sparse input)')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
    print(f'  {score}/{total} passed.' + ('' if score == total else ' Keep going!'))


_run_checks()

## Bonus Challenge

Right now the function sends the same two fixed example pairs for every call. On Day 13 (RAG I) you will learn about **retrieval-augmented generation**, where context is selected dynamically at runtime.

As a preview: try writing a `select_examples(text: str) -> list` function that picks the *most relevant* example pair based on simple keyword matching — if the input mentions a postcode-like token, prefer the fully-populated example; otherwise prefer the sparse one. Swap `select_examples(text)` in place of the hardcoded list and re-run the checks to confirm they still pass.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import json
import ollama
from pydantic import BaseModel, Field


class AddressInfo(BaseModel):
    street: str | None = Field(
        default=None,
        description="Street address including number and street name, or null if not present",
    )
    city: str | None = Field(
        default=None,
        description="City or town name, or null if not present",
    )
    state: str | None = Field(
        default=None,
        description="State, province, or region name, or null if not present",
    )
    country: str = Field(
        description="Country name — always present; infer from context if needed",
    )
    postcode: str | None = Field(
        default=None,
        description="Postal or ZIP code as a string, or null if not present",
    )


# Fabricated example pairs — never use real personal data in prompts.
_EXAMPLES: list[tuple[str, dict]] = [
    (
        # Example 1: all five fields present — shows the happy path
        "Ship the package to 14 Elmwood Avenue, Melbourne, Victoria 3000, Australia.",
        {
            "street": "14 Elmwood Avenue",
            "city": "Melbourne",
            "state": "Victoria",
            "country": "Australia",
            "postcode": "3000",
        },
    ),
    (
        # Example 2: only city and country — shows optional fields returning null
        "Our office is in Berlin, Germany.",
        {
            "street": None,
            "city": "Berlin",
            "state": None,
            "country": "Germany",
            "postcode": None,
        },
    ),
]


def extract_address_few_shot(text: str, model: str) -> dict:
    """Extract address fields from prose text using two few-shot example pairs.

    Builds a messages list containing:
      - A system message with the AddressInfo JSON Schema and extraction rules.
      - Two user/assistant demonstration pairs (fabricated, not real data).
      - A final user message containing the real text to extract from.

    Args:
        text:  The prose input to extract an address from.
        model: Ollama model name (e.g. "llama3.2").

    Returns:
        A plain dict with keys: street, city, state, country, postcode.
        Optional fields are None when absent in the text.

    Example:
        result = extract_address_few_shot(
            "Send the parcel to 42 Maple Drive, Austin, Texas 78701, USA.",
            model="llama3.2",
        )
        # result["city"] == "Austin"
        # result["postcode"] == "78701"
    """
    schema = AddressInfo.model_json_schema()

    system_content = (
        "Extract address information from the text.\n"
        "Return ONLY valid JSON matching this schema — no prose, no markdown:\n\n"
        f"{json.dumps(schema, indent=2)}\n\n"
        "If a field is not present in the text, set it to null."
    )

    messages: list[dict] = [{"role": "system", "content": system_content}]

    # Inject each fabricated example as a user/assistant demonstration pair
    for input_prose, output_dict in _EXAMPLES:
        messages.append({"role": "user", "content": input_prose})
        messages.append({"role": "assistant", "content": json.dumps(output_dict)})

    # Append the real extraction request
    messages.append({"role": "user", "content": text})

    response = ollama.chat(
        model=model,
        messages=messages,
        format="json",
    )

    address = AddressInfo.model_validate_json(response["message"]["content"])
    return address.model_dump()
```

**Why this works:** The two example pairs act as a concrete specification — the first shows the model what a fully populated extraction looks like (all five fields, correct value formats), while the second explicitly demonstrates that absent fields must be `null` rather than guessed or omitted. Because both patterns are shown *before* the real input arrives, the model infers the convention from the conversation structure rather than having to rely solely on field descriptions, which resolves the most common extraction errors in a single step.
</details>